# Ilaria — Faza 1 (corpus → tokenizare → antrenare pe H100)

Notebook autonom: codul e înglobat în celulele `%%writefile`, tokenizerul se antrenează aici (byte-level BPE, identic cu modul byte-level al motorului Go), corpusul se tokenizează în Python. Celulele 1–6 merg pe o sesiune **fără GPU**; 7–8 pe **H100**. Totul se scrie pe Drive în `MyDrive/ilaria`; o sesiune întreruptă se reia de unde a rămas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs('/content/nexus/forge', exist_ok=True)
os.makedirs('/content/drive/MyDrive/ilaria/corpus', exist_ok=True); os.makedirs('/content/drive/MyDrive/ilaria/brain-a', exist_ok=True)
%pip -q install datasets tokenizers
print('ok')

In [ ]:
import base64, io, zipfile
SRC = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(SRC))).extractall('/content/nexus/forge')
import os; print(sorted(os.listdir('/content/nexus/forge')))

## Tokenizer (5–10 min): eșantion RO/EN de 220 MB din HuggingFace → BPE byte-level 32k

Sare dacă `tokenizer.json` există deja în Drive.

In [ ]:
%%bash
cd /content/nexus
if [ -s /content/drive/MyDrive/ilaria/tokenizer.json ]; then echo 'tokenizer exists'; exit 0; fi
python -u forge/prepare_corpus.py --tokenizer-sample /content/tokenizer_sample.txt --sample-bytes 220000000 \
  --sources wiki_ro,fineweb2_ro,tinystories,fineweb_edu --max wiki_ro=100000 --max tinystories=80000 2>&1 | grep -v Warning
python forge/hf_tokenizer.py train --input /content/tokenizer_sample.txt --vocab 32000 --out /content/drive/MyDrive/ilaria/tokenizer.json
ls -la /content/drive/MyDrive/ilaria/tokenizer.json /content/drive/MyDrive/ilaria/tokenizer.hf.json

## Corpus pe Drive (1–3 ore, reluabil)

FineWeb-2 românesc (3 M documente), FineWeb-Edu (2 M), Wikipedia RO (300 k) și EN (500 k) ≈ 4 miliarde de tokeni.

In [ ]:
%%bash
cd /content/nexus
python -u forge/prepare_corpus.py --out-dir /content/drive/MyDrive/ilaria/corpus \
  --sources wiki_ro,wiki_en,fineweb2_ro,fineweb_edu \
  --max wiki_ro=300000 --max wiki_en=500000 --max fineweb2_ro=3000000 --max fineweb_edu=2000000 2>&1 | grep -v Warning
ls /content/drive/MyDrive/ilaria/corpus | head -60; du -sh /content/drive/MyDrive/ilaria/corpus

## Tokenizare (8 procese) + stream unic

Sare peste shard-urile care au deja `.bin`.

In [ ]:
%%bash
cd /content/nexus
for f in /content/drive/MyDrive/ilaria/corpus/*.jsonl; do p="${f%.jsonl}"; [ -f "$p.bin" ] || echo "$p"; done \
  | xargs -P 8 -I{} python forge/hf_tokenizer.py encode --tokenizer /content/drive/MyDrive/ilaria/tokenizer.json --in {}.jsonl --out {}
python forge/concat_streams.py --out /content/drive/MyDrive/ilaria/train_stream \
  --prefix ro=/content/drive/MyDrive/ilaria/corpus/fineweb2_ro --prefix ro=/content/drive/MyDrive/ilaria/corpus/wiki_ro \
  --prefix en=/content/drive/MyDrive/ilaria/corpus/fineweb_edu --prefix en=/content/drive/MyDrive/ilaria/corpus/wiki_en
cat /content/drive/MyDrive/ilaria/train_stream.json

## (H100) Viteza: 100 de pași cu rețeta A

Runtime → H100, apoi rulează din nou celulele de sus (montare, instalare, `%%writefile`), apoi aceasta.

In [ ]:
%%bash
cd /content/nexus
python forge/train_ilaria.py --data /content/drive/MyDrive/ilaria/train_stream --out /content/speedtest \
  --embed-dim 768 --heads 12 --layers 12 --ffn-dim 2688 --ctx 1024 --max-seq-len 1024 --rope --swiglu --dropout 0 \
  --batch 64 --accum 4 --steps 100 --warmup 20 --lr 6e-4 --min-lr 6e-5 --wd 0.1 --precision bf16 --compile --eval-every 100 2>&1 | tail -15

## (H100) Rețeta A — Ilaria-130M, până la 3 G tokeni

Checkpoint pe Drive la fiecare 500 de pași; dacă sesiunea cade, rulează din nou celula: reia din `checkpoint.pt`.

In [ ]:
%%bash
cd /content/nexus
TOK=$(python -c "import json;print(json.load(open('/content/drive/MyDrive/ilaria/train_stream.json'))['tokens'])")
STEPS=$(( TOK / 262144 )); [ $STEPS -gt 11500 ] && STEPS=11500
echo "tokens=$TOK steps=$STEPS"
RESUME=""; [ -f /content/drive/MyDrive/ilaria/brain-a/checkpoint.pt ] && RESUME="--resume /content/drive/MyDrive/ilaria/brain-a/checkpoint.pt"
python forge/train_ilaria.py --data /content/drive/MyDrive/ilaria/train_stream --out /content/drive/MyDrive/ilaria/brain-a \
  --embed-dim 768 --heads 12 --layers 12 --ffn-dim 2688 --ctx 1024 --max-seq-len 1024 --rope --swiglu --dropout 0 \
  --batch 64 --accum 4 --steps $STEPS --warmup 500 --lr 6e-4 --min-lr 6e-5 --wd 0.1 --precision bf16 --compile --eval-every 500 $RESUME

## Rezultatul

`/content/drive/MyDrive/ilaria/brain-a/transformer.nxtf` + `tokenizer.json` → pe PC în `data/forge/brain-a/`, apoi `go run -tags gpu ./cmd/nxtf-run -data-dir ./data/forge/brain-a -gpu -prompt "Ștefan cel Mare a fost"`.